<a href="https://colab.research.google.com/github/acapodanno/Large-Language-Model/blob/main/rag_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Setup delle dipendenze: Installare e configurare chromadb, sentence-transformers, rank-bm25 e langchain per l'ecosistema completo**

In [134]:
!pip install langchain langchain-community rank_bm25  faiss-cpu

In [135]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader_txt = TextLoader("./solar_system.txt",encoding="utf-8")
doc_txt = loader_txt.load()


#text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,separators=["\n\n", "\n", ". ", " ", "","**"])
#txt_chunks = text_splitter.split_documents(doc_txt)


### **Architettura modulare: Progettare classi base astratte per chunking, permettendo estensibilità futura con nuove strategie**

In [136]:
from abc import ABC, abstractmethod
from typing import List
from langchain_core.documents import Document

class BaseChunker(ABC):
    def __init__(self, chunk_size: int, chunk_overlap: int):
      self.chunk_size = chunk_size
      self.chunk_overlap = chunk_overlap

    @abstractmethod
    def chunk_documents(self, documents: List[Document]) -> List[Document]:
      pass

    def __repr__(self) -> str:
       return super().__repr__()


In [137]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
class RecursiveChunckerStrategy(BaseChunker):
  def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        super().__init__(chunk_size, chunk_overlap)
        self.name="RecursiveCharacterTextSplitter"
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
  def chunk_documents(self, documents: List[Document]) -> List[Document]:
    print(f"Esecuzione chunking con {self.name}...")
    return self._splitter.split_documents(doc_txt)

In [138]:
rcs = RecursiveChunckerStrategy(chunk_size=500, chunk_overlap=50)
txt_chunks = rcs.chunk_documents(doc_txt)

Esecuzione chunking con RecursiveCharacterTextSplitter...


In [139]:
from langchain_community.retrievers import BM25Retriever


retriever_bm25 = BM25Retriever.from_documents(
    documents=txt_chunks,

)
retriever_bm25.k = 3

In [140]:
retriever_bm25.invoke("Sole")

[Document(metadata={'source': './solar_system.txt'}, page_content='**La Terra**\nIl terzo pianeta dal Sole è il nostro, e merita un capitolo a sé stante (vedi sezione 4).'),
 Document(metadata={'source': './solar_system.txt'}, page_content='## 3. I Pianeti Interni (I Pianeti Terrestri)\nI quattro pianeti più vicini al Sole sono chiamati "terrestri" o "rocciosi" perché hanno una superficie solida composta principalmente da rocce e metalli.'),
 Document(metadata={'source': './solar_system.txt'}, page_content="* **La Nube di Oort:** Si ipotizza che esista un guscio sferico immenso, situato ai confini estremi dell'influenza gravitazionale del Sole (fino a 100.000 Unità Astronomiche di distanza). Questa nube è considerata la culla delle comete di lungo periodo.")]

In [141]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(txt_chunks, embeddings)
dense_retriever = vectorstore.as_retriever()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [142]:
dense_retriever.invoke("Mi dici il pianetta piú piccolo del sistema solare??")

[Document(id='084c8c9f-7c45-4a7b-a638-1cf76f1726e9', metadata={'source': './solar_system.txt'}, page_content="**Nettuno (Il Gigante Ghiacciato)**\nL'ottavo e ultimo pianeta ufficiale del Sistema Solare è Nettuno. Anch'esso un gigante ghiacciato di colore blu intenso (sempre a causa del metano), è il pianeta più ventoso del sistema, con venti supersonici che possono superare i 2.000 km/h. La sua luna principale, Tritone, orbita in direzione opposta alla rotazione di Nettuno ed espelle geyser di azoto liquido."),
 Document(id='b96c4387-e1f9-4573-b2c7-ee38ae640e92', metadata={'source': './solar_system.txt'}, page_content="**Saturno (Il Gigante Gassoso)**\nIl sesto pianeta è celebre per il suo vasto e brillante sistema di anelli, composti da innumerevoli particelle di ghiaccio e roccia, grandi da pochi micrometri a diversi metri. Saturno è il pianeta meno denso del Sistema Solare: la sua densità è inferiore a quella dell'acqua. La sua luna più interessante è Titano, l'unico satellite nel S

In [143]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, retriever_bm25],
    weights=[0.7, 0.3]
)

In [144]:
print("Ricerca 1: Testiamo una parola chiave/nome esatto")
query_1 = "Cos'è Sagittarius A* e dove si trova?"
risultati_1 = hybrid_retriever.invoke(query_1)
print(f"Risultato: {risultati_1[0].page_content}\n")

Ricerca 1: Testiamo una parola chiave/nome esatto
Risultato: **Il Centro Galattico**
Nel cuore della Via Lattea, a circa 26.000 anni luce dalla Terra, si trova una regione estremamente densa e attiva. Al centro esatto risiede *Sagittarius A\**, un buco nero supermassiccio con una massa pari a circa 4 milioni di volte quella del nostro Sole. Nonostante la sua immensa gravità, il buco nero non sta "risucchiando" l'intera galassia; le stelle e i sistemi planetari orbitano attorno a questo centro di massa proprio come i pianeti orbitano attorno al Sole.



In [145]:
print("Ricerca 2: Testiamo il significato (senza usare le parole esatte)")
query_2 = "Cosa fa da scudo al nostro pianeta impedendo che l'aria voli via nello spazio?"
risultati_2 = hybrid_retriever.invoke(query_2)
print(f"Risultato: {risultati_2[0].page_content}\n")

Ricerca 2: Testiamo il significato (senza usare le parole esatte)
Risultato: Il Sistema Solare si è formato circa 4,6 miliardi di anni fa dal collasso gravitazionale di una gigantesca nube molecolare di gas e polvere. La maggior parte della materia si è concentrata al centro, innescando le reazioni di fusione nucleare e dando vita al Sole. Il materiale residuo, ruotando attorno alla neonata stella, si è appiattito in un disco protoplanetario dal quale, per accrezione (collisione e aggregazione di frammenti), si sono formati i pianeti, le lune, gli asteroidi e le comete.



In [146]:
print("Ricerca 3: Testiamo l'azione combinata")
query_3 = "Da quali gas è composta la stella di classe spettrale G2V?"
risultati_3 = hybrid_retriever.invoke(query_3)
print(f"Risultato: {risultati_3[0].page_content}")

Ricerca 3: Testiamo l'azione combinata
Risultato: [Image of the solar system planets]


**Il Sole**
Il Sole è il cuore del nostro sistema. È una stella di classe spettrale G2V (spesso chiamata nana gialla) ed è composto principalmente da idrogeno (circa 73%) ed elio (circa 25%). Costituisce da solo il 99,86% di tutta la massa del Sistema Solare. La sua energia, generata dalla fusione nucleare nel nucleo, è la forza motrice che permette la vita sulla Terra e determina il clima e il meteo di tutti i pianeti.
